The goal of this notebook is to simply have very, very simple API calls into Saber using the API secret and API username, which are located here. Create the bearer token and make the calls just like we're using in the app. Keep it simple and start with the cell that says "Start here". 

In [1]:
import requests
import os

SABRE_API_USER_ID = os.environ.get("SABRE_API_USER_ID")

if not SABRE_API_USER_ID or SABRE_API_USER_ID in ["", "SABRE_API_USER_ID"]:
    raise Exception("API KEY MISSING OR INVALID. Please set your actual SABRE_API_USER_ID.")
else:
    print("All good! Valid key found.")

SABRE_API_SECRET = os.environ.get("SABRE_API_SECRET")
if not SABRE_API_SECRET or SABRE_API_SECRET in ["", "SABRE_API_SECRET"]:
    raise Exception("API KEY MISSING OR INVALID. Please set your actual SABRE_API_SECRET.")
else:
    print("All good! Valid key found.")


All good! Valid key found.
All good! Valid key found.


# Start Here

All Sabre calls follow the same two steps used in the app:

1. **Mint a bearer token** — `POST /v2/auth/token` (`grant_type=client_credentials`) with the
   `Authorization: Basic {secret}` header, where the secret is built from the two env vars above:

   ```
   secret = base64( base64(SABRE_API_USER_ID) + ":" + base64(SABRE_API_SECRET) )
   ```

2. **Call an endpoint** with `Authorization: Bearer {access_token}`.

Everything below runs against **CERT** (`https://api.cert.platform.sabre.com`) with the hackathon
credentials. Source of truth for entitlements: `specs/2026-07-13-sabre-cert-exploration/sabre-cert-notes.md`
(verified-live 2026-07-13/14).

---

## API endpoints from Sabre that ARE used in the app

Every call first mints the bearer via `POST /v2/auth/token`.

1. **`GET /v1/shop/flights`** — **InstaFlights** flight search. The real search behind the voice
   agent's flight shopping and flight-repair re-shop (`api/concierge.py`, `api/repair_tools.py` →
   `sabre_client.instaflights_search`). Returns real airlines + real fares. Always sent with
   `onlineitinerariesonly=N` (`Y` triggers a CERT-side 500).
2. **`GET /v1/lists/supported/shop/flights/origins-destinations`** — **Supported markets**. The city
   pairs InstaFlights carries; the agent validates a route against this before searching so it can give
   an honest "we don't serve that route" instead of a mock swap (`api/concierge.py` →
   `sabre_client.supported_markets`).

_Wired in `real_client.py` but **mock in the demo** (CERT entitlement walls — no PNR is ever written):_
`POST /v5/offers/shop` (Bargain Finder Max — entitled but returns 0 itineraries on this PCC),
and the Booking-Management writes `POST /v1/trip/orders/{createBooking, cancelBooking, modifyBooking}`
(`createBooking` is `UNAUTHORIZED_ACCESS`). These fall back to the mock per-call.

## API endpoints callable with the same bearer that are NOT used in the app

All verified-live on CERT with these exact credentials (see the sweep matrix in the CERT notes):

- **`GET /v2/shop/flights/fares?origin=&destination=`** — Lead Price Calendar (lowest fare per date).
- **`GET /v2/shop/flights/fares?origin=&topdestinations=`** — Destination Finder.
- **`GET /v1/lists/supported/cities`** — supported cities.
- **`GET /v1/lists/utilities/airlines`** and **`GET /v1/lists/utilities/aircraft/equipment`** — airline / aircraft code lookups.
- **`POST /v1/trip/orders/getBooking`** and **`POST /v1/trip/orders/cancelBooking`** — authorized (the entitlement wall is create-side only).
- **`POST /v1/offers/flightShop`** / **`flightShopLite`** — entitled but content-empty (0 offers on this PCC).


In [2]:
# --- 1. Mint the bearer token (same recipe the app uses) ----------------------
import base64

CERT_BASE_URL = "https://api.cert.platform.sabre.com"


def build_basic_secret(user_id: str, password: str) -> str:
    """secret = base64( base64(user_id) + ":" + base64(password) )"""
    b64 = lambda s: base64.b64encode(s.encode()).decode()
    return b64(f"{b64(user_id)}:{b64(password)}")


def mint_token() -> str:
    secret = build_basic_secret(SABRE_API_USER_ID, SABRE_API_SECRET)
    resp = requests.post(
        f"{CERT_BASE_URL}/v2/auth/token",
        headers={
            "Authorization": f"Basic {secret}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
        data={"grant_type": "client_credentials"},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


TOKEN = mint_token()
AUTH = {"Authorization": f"Bearer {TOKEN}"}
print(f"Token minted: {len(TOKEN)} chars, prefix {TOKEN[:6]}...")  # never print the whole token


Token minted: 376 chars, prefix T1RLAQ...


## Used endpoint #1 — InstaFlights flight search (`GET /v1/shop/flights`)

The real search behind the agent. `onlineitinerariesonly=N` is mandatory (`Y` → CERT 500).
InstaFlights is cache-based per city-pair, so pick a near-term-cached route/date — JFK→LAX at ~+2 days
was a verified hit. An empty cache window returns HTTP 404 `WARN.RAF.APPLICATION` ("No results were
found"), which the app treats as an honest "no flights", not an error.

`describe_itinerary()` pulls the **voice-ready** fields out of each priced itinerary
(`it["AirItinerary"]["OriginDestinationOptions"]["OriginDestinationOption"][0]`):

| Field | Path (relative to the itinerary) |
|---|---|
| Airline name | `FlightSegment[0].MarketingAirline.Code` → `GET /v1/lists/utilities/airlines?airlinecode=` |
| Flight number | `FlightSegment[0].FlightNumber` |
| Depart date/time | `FlightSegment[0].DepartureDateTime` (airport-local, no offset) |
| Arrive date/time | `FlightSegment[-1].ArrivalDateTime` (last segment; flags next-day / red-eye) |
| Total duration | `OriginDestinationOption[0].ElapsedTime` minutes → `Xh Ym` (whole journey incl. layovers) |
| Stops | non-stop when `(len(segments) - 1) + Σ FlightSegment[].StopQuantity == 0`; else loop segments for the layover airports |
| Cabin class | `AirItineraryPricingInfo.FareInfos.FareInfo[0].TPA_Extensions.Cabin.Cabin` → mapped (`Y`→Economy, `J`/`C`→Business, `F`→First) |
| Fare | `AirItineraryPricingInfo.ItinTotalFare.TotalFare` (`Amount` + `CurrencyCode`) |

In [3]:
# --- Used #1: InstaFlights search + voice-ready field extraction --------------
from datetime import date, timedelta, datetime

# Cabin code -> conversational cabin name (map the airline booking-class letter).
CABIN_NAMES = {
    "Y": "Economy", "S": "Economy", "B": "Economy", "M": "Economy",
    "W": "Premium Economy",
    "C": "Business", "J": "Business", "D": "Business", "I": "Business",
    "F": "First Class", "A": "First Class", "P": "First Class",
}

# Airline name lookup via /v1/lists/utilities/airlines, cached per code.
_airline_names = {}
def airline_name(code: str) -> str:
    if code not in _airline_names:
        r = requests.get(f"{CERT_BASE_URL}/v1/lists/utilities/airlines",
                         headers=AUTH, params={"airlinecode": code}, timeout=30)
        info = r.json().get("AirlineInfo", []) if r.status_code == 200 else []
        _airline_names[code] = info[0]["AirlineName"] if info else code
    return _airline_names[code]

def fmt_duration(minutes: int) -> str:      # 349 -> "5h 49m"
    return f"{minutes // 60}h {minutes % 60}m"

def fmt_when(iso: str) -> str:              # "2026-07-18T07:00:00" -> "Sat Jul 18, 7:00 AM"
    return datetime.fromisoformat(iso).strftime("%a %b %-d, %-I:%M %p")

def describe_itinerary(it: dict) -> None:
    opt = it["AirItinerary"]["OriginDestinationOptions"]["OriginDestinationOption"][0]
    segs = opt["FlightSegment"]
    first, last = segs[0], segs[-1]
    fare = it["AirItineraryPricingInfo"]["ItinTotalFare"]["TotalFare"]
    cabin = it["AirItineraryPricingInfo"]["FareInfos"]["FareInfo"][0]["TPA_Extensions"]["Cabin"]["Cabin"]

    # 1. Logistics & timing (whole-journey figures)
    carrier = first["MarketingAirline"]["Code"]
    print(f"{airline_name(carrier)} ({carrier}) flight {first['FlightNumber']}")
    print(f"  Depart:   {fmt_when(first['DepartureDateTime'])}  {first['DepartureAirport']['LocationCode']}")
    overnight = last["ArrivalDateTime"][:10] != first["DepartureDateTime"][:10]
    print(f"  Arrive:   {fmt_when(last['ArrivalDateTime'])}  {last['ArrivalAirport']['LocationCode']}"
          + ("  (arrives next day)" if overnight else ""))
    print(f"  Duration: {fmt_duration(opt['ElapsedTime'])}")   # option-level = full journey incl. layovers

    # 2. Stops vs non-stop  (connections + any direct-flight stops)
    total_stops = (len(segs) - 1) + sum(s.get("StopQuantity", 0) for s in segs)
    if total_stops == 0:
        print("  Stops:    Non-stop")
    else:
        vias = ", ".join(s["ArrivalAirport"]["LocationCode"] for s in segs[:-1])
        print(f"  Stops:    {total_stops} stop(s) via {vias}")
        for s in segs:
            print(f"    - {s['MarketingAirline']['Code']}{s['FlightNumber']} "
                  f"{s['DepartureAirport']['LocationCode']}->{s['ArrivalAirport']['LocationCode']} "
                  f"dep {fmt_when(s['DepartureDateTime'])}")

    # 3. Cabin class
    print(f"  Cabin:    {CABIN_NAMES.get(cabin, cabin)} ('{cabin}')")
    print(f"  Fare:     {fare['Amount']} {fare['CurrencyCode']}")

# InstaFlights is a per-pair cache; JFK->LAX at ~+2 days was live on 2026-07-16.
# If it 404s (WARN.RAF.APPLICATION), the cache window shifted — try another pair
# from the supported-markets list below, or another near-term date.
depart = (date.today() + timedelta(days=2)).isoformat()
resp = requests.get(
    f"{CERT_BASE_URL}/v1/shop/flights",
    headers=AUTH,
    params={
        "origin": "JFK",
        "destination": "LAX",
        "departuredate": depart,
        "onlineitinerariesonly": "N",   # Y -> CERT 500
        "limit": 3,
    },
    timeout=30,
)
print("HTTP", resp.status_code, "| route JFK -> LAX on", depart, "\n")

if resp.status_code == 200:
    itins = resp.json().get("PricedItineraries", [])
    print(f"{len(itins)} priced itineraries\n")
    for it in itins:
        describe_itinerary(it)
        print()
else:
    print(resp.json())   # 404 WARN.RAF.APPLICATION = empty cache window (honest "no flights")


HTTP 200 | route JFK -> LAX on 2026-07-18 

3 priced itineraries

Delta Air Lines, Inc. (DL) flight 713
  Depart:   Sat Jul 18, 7:00 AM  JFK
  Arrive:   Sat Jul 18, 9:49 AM  LAX
  Duration: 5h 49m
  Stops:    Non-stop
  Cabin:    Economy ('Y')
  Fare:     341.4 USD

Delta Air Lines, Inc. (DL) flight 773
  Depart:   Sat Jul 18, 9:00 AM  JFK
  Arrive:   Sat Jul 18, 11:54 AM  LAX
  Duration: 5h 54m
  Stops:    Non-stop
  Cabin:    Economy ('Y')
  Fare:     341.4 USD

Delta Air Lines, Inc. (DL) flight 767
  Depart:   Sat Jul 18, 11:00 AM  JFK
  Arrive:   Sat Jul 18, 1:49 PM  LAX
  Duration: 5h 49m
  Stops:    Non-stop
  Cabin:    Economy ('Y')
  Fare:     341.4 USD



## Used endpoint #2 — Supported markets (`GET /v1/lists/supported/shop/flights/origins-destinations`)

The city pairs InstaFlights carries. The agent checks a requested route against this list before
searching (probed with `destinationcountry=US`, the demo's domestic routes).

In [4]:
# --- Used #2: supported markets -----------------------------------------------
resp = requests.get(
    f"{CERT_BASE_URL}/v1/lists/supported/shop/flights/origins-destinations",
    headers=AUTH,
    params={"destinationcountry": "US"},
    timeout=30,
)
print("HTTP", resp.status_code)
pairs = resp.json().get("OriginDestinationLocations", [])
print(f"{len(pairs)} supported city pairs (first 5):")
for p in pairs[:5]:
    print(f"  {p['OriginLocation']['AirportCode']} -> {p['DestinationLocation']['AirportCode']}")


HTTP 200
731 supported city pairs (first 5):
  ABE -> MCO
  AMS -> LAX
  AMS -> NYC
  ARN -> LAX
  ARN -> MIA


## Bonus — endpoints callable with the same bearer but NOT used in the app

Same `AUTH` bearer, all verified-live on CERT. The app doesn't call these; they're here to show the
bearer's reach. Lead Price Calendar (fares from today ~6 months out) is the most useful unused one.

In [5]:
# --- Not used in the app, but callable with the same bearer -------------------

# Lead Price Calendar: lowest fare per date for a city pair (needs departuredate + lengthofstay)
d30 = (date.today() + timedelta(days=30)).isoformat()
resp = requests.get(
    f"{CERT_BASE_URL}/v2/shop/flights/fares",
    headers=AUTH,
    params={"origin": "DFW", "destination": "LAX", "departuredate": d30, "lengthofstay": 5},
    timeout=30,
)
fares = resp.json().get("FareInfo", []) if resp.status_code == 200 else []
print(f"Lead Price Calendar  HTTP {resp.status_code}  ({len(fares)} dated fares)")
for f in fares[:3]:
    print(f"  {f.get('DepartureDateTime', '')[:10]}  lowest {f.get('LowestFare', {}).get('Fare')} "
          f"{f.get('LowestFare', {}).get('AirlineCodes')}")

# Destination Finder: cheapest top destinations from an origin
resp = requests.get(
    f"{CERT_BASE_URL}/v2/shop/flights/fares",
    headers=AUTH,
    params={"origin": "DFW", "lengthofstay": 5, "topdestinations": 5},
    timeout=30,
)
print(f"\nDestination Finder   HTTP {resp.status_code}  "
      f"({len(resp.json().get('FareInfo', [])) if resp.status_code == 200 else 0} destinations)")

# Airline code lookup
resp = requests.get(
    f"{CERT_BASE_URL}/v1/lists/utilities/airlines",
    headers=AUTH,
    params={"airlinecode": "AA"},
    timeout=30,
)
print(f"Airline lookup (AA)  HTTP {resp.status_code}  "
      f"{resp.json() if resp.status_code == 200 else resp.text[:120]}")


Lead Price Calendar  HTTP 200  (1 dated fares)
  2026-08-15  lowest 344.8 ['DL']



Destination Finder   HTTP 200  (155 destinations)
Airline lookup (AA)  HTTP 200  {'AirlineInfo': [{'AirlineCode': 'AA', 'AirlineName': 'American Airlines', 'AlternativeBusinessName': 'American Airlines'}], 'Links': [{'rel': 'linkTemplate', 'href': 'https://api.cert.platform.sabre.com/v1/lists/utilities/airlines?airlinecode=<airlinecode>'}, {'rel': 'self', 'href': 'https://api.cert.platform.sabre.com/v1/lists/utilities/airlines?airlinecode=AA'}]}


## Hotels part 1 — the classic (CSL) hotel REST family: dead on these credentials

> **UPDATE (2026-07-16 evening):** Sabre's hackathon collection exposes a NEWER *agentic* hotel API
> family that IS live with this bearer — see **Hotels part 2** two cells down. This section documents
> the classic family only; the real blocker turned out to be PCC authorization, not missing endpoints.

Full sweep of Sabre's classic hotel REST family on CERT with the same bearer (probe matrix in the
next cell). **None of these endpoints return hotel data.**

| Endpoint family | Best result | What it means |
|---|---|---|
| **Get Hotel Avail** `POST /v5/get/hotelavail` | `400 VALIDATION_FAILED` ("matched 0 out of 3 schemas") with the **exact JSON body developer.sabre.com documents**; a stripped GeoSearch-only body passes validation but then dies in the backend (`NGHP-DISTRIBUTION.INTERNAL_ERROR` at `convertToOutputFormat`) | Route **is entitled** — the bearer reaches schema validation and the backend. But CERT's deployed schema rejects `RateInfoRef` (stay dates / rooms / currency) in every branch, and the backend errors on the shapes it does accept. No rates retrievable either way. |
| Get Hotel Avail v1 / v2 / v4 | `404` `ERR.2SG.CLIENT.INVALID_REQUEST` ("not found in rest table") | Older REST versions not provisioned for this app |
| Get Hotel Avail v3, Details v2, Content v2 | `400 SERVICE_VERSION_DEPRECATED` | Versions retired by Sabre |
| **Get Hotel Details** `POST /v5/get/hoteldetails` | `400 VALIDATION_FAILED` ("matched 0 out of 2") | Same story as Avail v5 — entitled, but schema wall |
| Hotel Content / Media (`/v1.0.0/shop/hotels/*`) | `404` "No service exists" / `500` | Not deployed on CERT |
| Get Hotel Media (`/v1/get/hotelmedia`) | empty-body `404` with a `GetHotelMediaRQ` payload; `403 ERR.2SG.SEC.NOT_AUTHORIZED` with anything else | Nothing usable behind the route for this credential |
| GeoSearch v1 (`/v1/lists/utilities/geosearch/locations`) | `400 ERR.RAF.VALIDATION` — engine wants the geosearch **v3 XML** document (`Around` / `ForPlaces`), not the JSON the v2/v4 docs show; v2 itself is `404` | Location lookup only (no rates), and the deployed version doesn't match its docs either |

Control on the same bearer: flight endpoints (`/v1/shop/flights`, supported markets, fares) all answer
`200` — none of this is a token or auth problem.

In [6]:
# --- Hotels: probe matrix — every hotel endpoint fails on these credentials ----
checkin = (date.today() + timedelta(days=14)).isoformat()
checkout = (date.today() + timedelta(days=16)).isoformat()

GEO = {"GeoRef": {"Radius": 20, "UOM": "MI",
                  "RefPoint": {"Value": "LAX", "ValueContext": "CODE", "RefPointType": "6"}}}

# The exact REST body developer.sabre.com documents for Get Hotel Avail v5 — CERT rejects it.
AVAIL_DOCUMENTED = {"GetHotelAvailRQ": {"SearchCriteria": {
    "OffSet": 1, "SortBy": "TotalRate", "SortOrder": "ASC", "PageSize": 5,
    "GeoSearch": GEO,
    "RateInfoRef": {"CurrencyCode": "USD", "BestOnly": "1",
                    "StayDateRange": {"StartDate": checkin, "EndDate": checkout},
                    "Rooms": {"Room": [{"Index": 1, "Adults": 1}]}}}}}

# The only shape CERT's v5 schema accepts (no dates/rooms allowed!) — backend then errors anyway.
AVAIL_GEO_ONLY = {"GetHotelAvailRQ": {"SearchCriteria": {"GeoSearch": GEO}}}

PROBES = [
    ("Avail v5 - documented body", "/v5/get/hotelavail", AVAIL_DOCUMENTED),
    ("Avail v5 - GeoSearch only",  "/v5/get/hotelavail", AVAIL_GEO_ONLY),
    ("Avail v4",                   "/v4/get/hotelavail", AVAIL_DOCUMENTED),
    ("Avail v3",                   "/v3/get/hotelavail", AVAIL_DOCUMENTED),
    ("Details v5",                 "/v5/get/hoteldetails",
     {"GetHotelDetailsRQ": {"SearchCriteria": {
         "HotelRefs": {"HotelRef": {"HotelCode": "100005565"}},
         "RateInfoRef": {"CurrencyCode": "USD",
                         "StayDateRange": {"StartDate": checkin, "EndDate": checkout},
                         "Rooms": {"Room": [{"Index": 1, "Adults": 1}]}}}}}),
    ("Content (shop/hotels)",      "/v1.0.0/shop/hotels/content", {"GetHotelContentRQ": {}}),
    ("Media v1",                   "/v1/get/hotelmedia", {"GetHotelMediaRQ": {}}),
    ("GeoSearch v1 (hotels @LAX)", "/v1/lists/utilities/geosearch/locations",
     {"GeoSearchRQ": {"GeoRef": {"Radius": 10, "UOM": "MI", "Category": "HOTEL",
                                 "RefPoint": {"Value": "LAX", "ValueContext": "CODE",
                                              "RefPointType": "6"}}}}),
]

VERDICTS = {  # errorCode -> what it actually means
    "ERR.NGHP-DISTRIBUTION.CLIENT.VALIDATION_FAILED": "entitled, but CERT schema rejects the documented body",
    "ERR.NGHP-DISTRIBUTION.INTERNAL_ERROR": "passed validation; hotel backend itself errors",
    "ERR.2SG.CLIENT.INVALID_REQUEST": "version not provisioned for this app",
    "ERR.2SG.SERVICE_VERSION_DEPRECATED": "version retired by Sabre",
    "ERR.2SG.SEC.NOT_AUTHORIZED": "endpoint exists; credential not entitled",
    "WARN.RAF.APPLICATION": "no such service deployed on CERT",
    "ERR.RAF.VALIDATION": "engine wants the geosearch v3 XML document, not this JSON",
    "": "empty 404 body — nothing behind the route (403 NOT_AUTHORIZED with other payloads)",
}

hotel_data_seen = False
for name, path, body in PROBES:
    r = requests.post(f"{CERT_BASE_URL}{path}",
                      headers={**AUTH, "Content-Type": "application/json"},
                      json=body, timeout=60)
    try:
        code = r.json().get("errorCode", "")
    except ValueError:
        code = ""
    if r.status_code == 200:
        hotel_data_seen = True
        print(f"{name:28} HTTP 200  <- WORKS NOW — update the summary above!")
    else:
        print(f"{name:28} HTTP {r.status_code}  {code}")
        print(f"{'':28} -> {VERDICTS.get(code, 'unclassified - inspect r.text')}")

print("\nVerdict:", "a hotel endpoint returned data — revisit the mock!" if hotel_data_seen
      else "no hotel data retrievable on CERT — the app's hotel leg stays mock (matches sabre-cert-notes.md)")

Avail v5 - documented body   HTTP 400  ERR.NGHP-DISTRIBUTION.CLIENT.VALIDATION_FAILED
                             -> entitled, but CERT schema rejects the documented body


Avail v5 - GeoSearch only    HTTP 400  ERR.NGHP-DISTRIBUTION.INTERNAL_ERROR
                             -> passed validation; hotel backend itself errors


Avail v4                     HTTP 404  ERR.2SG.CLIENT.INVALID_REQUEST
                             -> version not provisioned for this app
Avail v3                     HTTP 400  ERR.2SG.SERVICE_VERSION_DEPRECATED
                             -> version retired by Sabre


Details v5                   HTTP 400  ERR.NGHP-DISTRIBUTION.CLIENT.VALIDATION_FAILED
                             -> entitled, but CERT schema rejects the documented body
Content (shop/hotels)        HTTP 404  WARN.RAF.APPLICATION
                             -> no such service deployed on CERT


Media v1                     HTTP 404  
                             -> empty 404 body — nothing behind the route (403 NOT_AUTHORIZED with other payloads)


GeoSearch v1 (hotels @LAX)   HTTP 400  ERR.RAF.VALIDATION
                             -> engine wants the geosearch v3 XML document, not this JSON

Verdict: no hotel data retrievable on CERT — the app's hotel leg stays mock (matches sabre-cert-notes.md)


## Hotels part 2 — the agentic Hotel APIs ARE live; the only blocker is credential authorization

Sabre's hackathon collection points to a newer **agentic (LLM-friendly) hotel API family** — the same
APIs the Sabre MCP server wraps. These are real, deployed on CERT, and callable with our existing
bearer at base path **`https://api.cert.platform.sabre.com/v1/hotels`** (from the OpenAPI spec at
`developer.sabre.com/rest-api/mcp-hotel-search-api`):

| API | Endpoint | Request (flat JSON) |
|---|---|---|
| Hotel Search | `POST /v1/hotels/hotelSearch` | `radiusInMiles`, `checkInDate`, `checkOutDate`, `numberOfAdults`, plus ONE of `latitude`+`longitude` / `referencePoint` (`{"type": "Airport", "value": "DFW"}`) / `address`; optional `searchSource`, `maxResults`, `pos` |
| Hotel Rates | `POST /v1/hotels/getHotelRates` | `hotelCode`, `checkInDate`, `checkOutDate`, `numberOfAdults` |
| Hotel Price Check | `POST /v1/hotels/checkHotelRate` | `hotelPriceCheckRq.rateInfoRef.rateKey` (rateKey comes from a rates response) |

Every call returns **HTTP 200 with structured JSON** (`{timestamp, warnings/errors, hotels/rooms}`) —
the API family works. What's missing is *content authorization*, and the errors say exactly why:

| Attempt | Provider verdict | Meaning |
|---|---|---|
| Rates for a known CERT test hotel (`100072188` Hyatt Regency Tulsa) | `INVALID_PCC._LENGTH_OF_PCC_IS_GREATER_THAN_5_CHARACTERS.` | **The root cause, named explicitly:** our hackathon token's principal is an app ID, not a real agency PCC, so GDS hotel content rejects it |
| Search, default sources (includes GDS) | `ERROR_DURING_PROCESSING,_PLEASE_RETRY` (persists across retries) | Same PCC problem — search just masks it as a generic retryable error |
| Search with `"pos": {"source": {"pseudoCityCode": "S5OM"}}` | `NOT_AUTHORIZED_TO_SWITCH_TO_S5OM` | `pos` is the documented branch-switch into another PCC. Sabre confirmed on Discord that **hotels are enabled in PCC S5OM**, but our app is not on that branch's authorized list (also tried the `S50M` spelling — same wall) |
| Search with `searchSource: ALL_NON_GDS` / `EXPEDIA` | `NO_HOTELS_FOUND_WHICH_MATCH_THIS_INPUT.` | Clean round-trip! Aggregator sources simply have **zero CERT inventory** for us (probed DFW / NYC / LAS / MCO / London / Cancun at +30 and +90 days) |

### The Sabre MCP server (`https://mcp2.cert.sabre.com/mcp`) — same wall (part 2b cell below)

Discord (Saptanshu, 7/15) said this URL "should now work for both air and hotels." Verified 7/16 with
our token: the server is **up and speaks proper MCP OAuth discovery** (unauthenticated `initialize` →
`401` with a `resource_metadata` pointer), but our bearer gets **`403 insufficient_scope`** ("the
request requires higher privileges than provided by the access token"). Re-minting with `resource=` /
`audience=` / `scope=` params changes nothing, and the MCP host's advertised own token endpoint 404s —
so there's no self-serve path to higher privileges. The MCP inherits the credential's entitlements;
it is not a separate content channel.

**Bottom line for the team:** every Sabre hotel door — classic REST, agentic REST, and the MCP server —
fails on the *same single cause*: our hackathon credential lacks hotel authorization (invalid PCC
principal + not allowed to branch into S5OM + insufficient scope for MCP). Working hotel access for
other teams proves the content exists. The one-sentence ask for the Discord thread: **"Our app gets
`NOT_AUTHORIZED_TO_SWITCH_TO_S5OM` on `/v1/hotels/hotelSearch` and `403 insufficient_scope` on
`mcp2.cert.sabre.com/mcp` — can you enable hotel/S5OM access for our application credentials?"**
Zero code changes needed on our side once flipped: `hotelSearch` → `getHotelRates` → `checkHotelRate`
(or the MCP tools) is a complete voice-ready shop flow, and the probe cells below will light up with
real hotels. Until then the app's hotel leg stays mock.

In [7]:
# --- Hotels part 2: agentic Hotel APIs — live, blocked only on PCC ------------
# Base path from the OpenAPI spec (developer.sabre.com -> rest-api/mcp-hotel-search-api):
#   https://api.cert.platform.sabre.com/v1/hotels
ci = (date.today() + timedelta(days=30)).isoformat()
co = (date.today() + timedelta(days=32)).isoformat()


def agentic(path, body, label):
    r = requests.post(f"{CERT_BASE_URL}/v1/hotels{path}",
                      headers={**AUTH, "Content-Type": "application/json"},
                      json=body, timeout=120)
    out = r.json() if "json" in r.headers.get("content-type", "") else {}
    results = out.get("hotels") or out.get("rooms") or []
    problems = out.get("warnings") or out.get("errors") or []
    ptype = problems[0].get("type", "") if problems else ""
    print(f"{label:36} HTTP {r.status_code}  results={len(results)}  {ptype}")
    if results:
        print(f"{'':36} first: {results[0]}")
    return out


SEARCH = {"radiusInMiles": 50, "checkInDate": ci, "checkOutDate": co,
          "numberOfAdults": 2, "maxResults": 5, "currencyCode": "USD",
          "referencePoint": {"type": "Airport", "value": "DFW"}}

# 1. Default (ALL sources incl. GDS): a persistent "please retry" that never clears —
#    the GDS leg can't shop from our principal (see #4 for the explicit reason).
agentic("/hotelSearch", SEARCH, "hotelSearch (default sources)")

# 2. Aggregator-only content: API round-trips cleanly, but CERT has zero non-GDS
#    inventory for us (probed DFW/NYC/LAS/MCO/London/Cancun at +30d and +90d).
agentic("/hotelSearch", {**SEARCH, "searchSource": "ALL_NON_GDS"},
        "hotelSearch ALL_NON_GDS")

# 3. Branch-switch into the hotel-enabled PCC Sabre named on Discord (S5OM):
#    the documented pos.source.pseudoCityCode mechanism — our app isn't authorized (yet).
agentic("/hotelSearch", {**SEARCH, "pos": {"source": {"pseudoCityCode": "S5OM"}}},
        "hotelSearch via PCC S5OM")

# 4. Rates for a known CERT test property (Hyatt Regency Tulsa, from Sabre's docs):
#    this one names the root cause — our token's principal is an app ID, not a
#    valid agency PCC ("length of PCC > 5 characters").
agentic("/getHotelRates", {"currencyCode": "USD", "checkInDate": ci,
                           "checkOutDate": co, "hotelCode": "100072188",
                           "numberOfAdults": 2},
        "getHotelRates (Hyatt Rgcy Tulsa)")

hotelSearch (default sources)        HTTP 200  results=0  ERROR_DURING_PROCESSING,_PLEASE_RETRY


hotelSearch ALL_NON_GDS              HTTP 200  results=0  NO_HOTELS_FOUND_WHICH_MATCH_THIS_INPUT.


hotelSearch via PCC S5OM             HTTP 200  results=0  NOT_AUTHORIZED_TO_SWITCH_TO_S5OM


getHotelRates (Hyatt Rgcy Tulsa)     HTTP 200  results=0  INVALID_PCC._LENGTH_OF_PCC_IS_GREATER_THAN_5_CHARACTERS.


{'timestamp': '2026-07-16T20:49:50.828714113Z',
 'errors': [{'category': 'APPLICATION',
   'type': 'INVALID_PCC._LENGTH_OF_PCC_IS_GREATER_THAN_5_CHARACTERS.',
   'description': 'Provider message: '}]}

In [8]:
# --- Hotels part 2b: the Sabre MCP server itself (mcp2.cert.sabre.com) --------
# Discord (Saptanshu, 7/15): "https://mcp2.cert.sabre.com/mcp should now work for
# both air and hotels." Verdict with OUR token: the endpoint is real and healthy,
# but rejects us with 403 insufficient_scope — same authorization wall as REST.
MCP_URL = "https://mcp2.cert.sabre.com/mcp"
init = {"jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {"protocolVersion": "2025-03-26", "capabilities": {},
                   "clientInfo": {"name": "vocal-bridge-probe", "version": "0.1"}}}

# 1. No auth -> 401 + proper MCP OAuth discovery pointer (so the server is up)
r = requests.post(MCP_URL, json=init,
                  headers={"Accept": "application/json, text/event-stream"}, timeout=30)
print(f"initialize, no auth:    HTTP {r.status_code}")
print(f"  www-authenticate: {r.headers.get('www-authenticate', '')[:90]}")

# 2. Our platform bearer -> 403 insufficient_scope. Also tried mints with
#    resource= / audience= / scope= params on /v2/auth/token — identical result.
r = requests.post(MCP_URL, json=init,
                  headers={"Accept": "application/json, text/event-stream",
                           "Content-Type": "application/json", **AUTH}, timeout=30)
print(f"initialize, our bearer: HTTP {r.status_code}")
print(f"  www-authenticate: {r.headers.get('www-authenticate', '')[:130]}")

# 3. OAuth discovery metadata (public). The advertised MCP-local token endpoint
#    404s, so there is no self-serve way to mint a higher-privilege token —
#    Sabre has to entitle the credential itself.
r = requests.get("https://mcp2.cert.sabre.com/.well-known/oauth-authorization-server",
                 timeout=30)
meta = r.json()
print(f"advertised token endpoint: {meta.get('token_endpoint')}")
r = requests.post(meta["token_endpoint"],
                  headers={"Authorization": f"Basic {build_basic_secret(SABRE_API_USER_ID, SABRE_API_SECRET)}",
                           "Content-Type": "application/x-www-form-urlencoded"},
                  data={"grant_type": "client_credentials"}, timeout=30)
print(f"minting there:             HTTP {r.status_code} (404 = not actually deployed)")

initialize, no auth:    HTTP 401
  www-authenticate: Bearer resource_metadata=https://mcp2.cert.sabre.com/.well-known/oauth-protected-resource/


initialize, our bearer: HTTP 403
  www-authenticate: Bearer error="insufficient_scope", error_description="The request requires higher privileges than provided by the access token.", 
advertised token endpoint: https://mcp2.cert.sabre.com/v2/auth/token


minting there:             HTTP 404 (404 = not actually deployed)
